# 05. 최종 평가 및 오류 분석

**test split을 사용하는 유일한 노트북입니다.**

`01`~`04`에서 test는 한 번도 쓰이지 않았습니다. 모델 선택도, 하이퍼파라미터 조정도
전부 val로만 했습니다. 그래서 여기서 나오는 숫자는 **진짜 일반화 성능**입니다.

> 이 노트북을 여러 번 돌리면서 설정을 바꾸면 그 순간 test가 오염됩니다.
> 결과가 마음에 들지 않아도 되돌아가 조정하지 마세요. 그건 보고서에 쓸 수 없는 숫자가 됩니다.

**평가 대상**: `resnet18_finetune` (val macro-F1 0.9371로 최고)

**이 노트북에서 답할 질문**
1. test 성능이 val과 비슷한가 (과적합 여부)
2. 어떤 클래스 쌍이 서로 헷갈리는가
3. EDA에서 세운 가설(유리 3종 색상 분리)이 맞았는가
4. 모델이 틀릴 때 확신하고 틀리는가, 애매해하며 틀리는가

## 1. 환경 및 데이터

In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch import amp
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
import torchvision.models as models
from sklearn.metrics import (confusion_matrix, classification_report,
                             f1_score, precision_recall_fscore_support)

assert torch.cuda.is_available(), "CUDA 미탐지. 커널이 .venv인지 확인하세요."
device = torch.device("cuda")

sns.set_theme(style="whitegrid")
plt.rcParams["font.family"] = "Malgun Gothic"
plt.rcParams["axes.unicode_minus"] = False

OUT = Path("outputs/garbage")
BATCH_SIZE = 64

split     = pd.read_csv(OUT / "metrics" / "split.csv")
label_map = json.load(open(OUT / "metrics" / "label_map.json", encoding="utf-8"))
stats     = json.load(open(OUT / "metrics" / "norm_stats.json", encoding="utf-8"))

classes   = [c for c, _ in sorted(label_map.items(), key=lambda kv: kv[1])]
N_CLASSES = len(classes)
test_df   = split[split.split == "test"].reset_index(drop=True)

print(f"test {len(test_df):,}장 / {N_CLASSES}클래스")
print(test_df["label"].value_counts().sort_index().to_string())

## 2. 모델 복원

In [ ]:
CKPT = OUT / "models" / "transfer_finetune.pt"
ck = torch.load(CKPT, map_location=device)

print("체크포인트:", CKPT.name)
print("  선택 에폭     :", ck["epoch"])
print("  val macro_f1  :", round(ck["val_macro_f1"], 4))
print("  입력 크기     :", ck["img_size"])

IMG_SIZE = ck["img_size"]
assert ck["classes"] == classes, "클래스 순서 불일치 — label_map.json 확인 필요"

model = models.resnet18(weights=None)                 # 구조만 (가중치는 체크포인트에서)
model.fc = nn.Linear(model.fc.in_features, N_CLASSES)
model.load_state_dict(ck["model"])
model = model.to(device).eval()

mean, std = stats["imagenet_mean"], stats["imagenet_std"]

eval_tf = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(mean, std),
])


class GarbageDataset(Dataset):
    def __init__(self, df, transform):
        self.paths   = df["path"].tolist()
        self.targets = df["label_idx"].tolist()
        self.transform = transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, i):
        img = Image.open(self.paths[i]).convert("RGB")
        return self.transform(img), self.targets[i]


test_ds = GarbageDataset(test_df, eval_tf)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False,
                         num_workers=0, pin_memory=True)
print("\n모델 복원 완료")

## 3. test 추론 — 단 한 번

확률까지 저장합니다. 모델이 **얼마나 확신하며 틀렸는지**를 뒤에서 분석하기 위해서입니다.

In [ ]:
@torch.no_grad()
def predict(model, loader):
    probs, trues = [], []
    for inputs, labels in loader:
        inputs = inputs.to(device, non_blocking=True)
        with amp.autocast("cuda", dtype=torch.float16):
            logits = model(inputs)
        probs.append(F.softmax(logits.float(), dim=1).cpu())
        trues.append(labels)
    return torch.cat(probs).numpy(), torch.cat(trues).numpy()


probs, y_true = predict(model, test_loader)
y_pred = probs.argmax(1)
conf   = probs.max(1)                    # 예측 확신도

test_acc      = 100 * (y_pred == y_true).mean()
test_macro_f1 = f1_score(y_true, y_pred, average="macro")
test_w_f1     = f1_score(y_true, y_pred, average="weighted")

print(f"test accuracy    : {test_acc:.2f}%")
print(f"test macro-F1    : {test_macro_f1:.4f}")
print(f"test weighted-F1 : {test_w_f1:.4f}")
print(f"(참고) val macro-F1 : {ck['val_macro_f1']:.4f}  "
      f"차이 {test_macro_f1 - ck['val_macro_f1']:+.4f}")

**val과 test 차이 읽는 법**

±0.02 이내면 정상입니다. 모델 선택을 val로 했으니 val이 약간 높게 나오는 게 자연스럽습니다.

차이가 0.05 이상 벌어졌다면 val에 과적합된 것입니다. 체크포인트를 val macro-F1로
고르는 과정 자체가 val을 조금씩 소모하기 때문인데, 12에폭 정도면 심하게 나타나지 않습니다.

## 4. 클래스별 성능

In [ ]:
prec, rec, f1, support = precision_recall_fscore_support(
    y_true, y_pred, labels=range(N_CLASSES), zero_division=0)

per_class = pd.DataFrame({
    "class":     classes,
    "n":         support,
    "precision": prec.round(3),
    "recall":    rec.round(3),
    "f1":        f1.round(3),
}).sort_values("f1")

per_class.to_csv(OUT / "metrics" / "test_per_class.csv",
                 index=False, encoding="utf-8")
print(per_class.to_string(index=False))

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
order = per_class["class"].tolist()
x = np.arange(len(order))
w = 0.38

ax.bar(x - w/2, per_class["precision"], w, label="precision", color="#4a7ba7")
ax.bar(x + w/2, per_class["recall"],    w, label="recall",    color="#c0392b")
ax.axhline(test_macro_f1, ls="--", lw=1, color="gray",
           label=f"macro-F1 {test_macro_f1:.3f}")

ax.set_xticks(x); ax.set_xticklabels(order, rotation=45, ha="right")
ax.set_ylim(0, 1.05); ax.set_ylabel("score")
ax.set_title("Test precision / recall by class (sorted by F1)")
ax.legend()
plt.tight_layout()
plt.savefig(OUT / "figures" / "14_test_per_class.png", dpi=150)
plt.show()

**precision과 recall을 나눠 보는 이유**

- **recall이 낮다** = 그 클래스를 자주 놓친다 (다른 클래스로 잘못 보냄)
- **precision이 낮다** = 다른 클래스를 그 클래스로 잘못 끌어온다

같은 F1이라도 원인이 정반대입니다. 개선 방향도 달라집니다.

## 5. 혼동행렬

In [ ]:
cm  = confusion_matrix(y_true, y_pred, labels=range(N_CLASSES))
cmn = cm / cm.sum(axis=1, keepdims=True)     # 행 정규화 → 대각선 = recall

fig, axes = plt.subplots(1, 2, figsize=(19, 7))

sns.heatmap(cm, ax=axes[0], cmap="Blues", annot=True, fmt="d", square=True,
            xticklabels=classes, yticklabels=classes, cbar=False)
axes[0].set_title("Confusion matrix (counts)")

sns.heatmap(cmn, ax=axes[1], cmap="Blues", vmin=0, vmax=1, annot=True, fmt=".2f",
            square=True, xticklabels=classes, yticklabels=classes,
            annot_kws={"size": 7})
axes[1].set_title(f"Row-normalized (diagonal = recall) — macro-F1 {test_macro_f1:.3f}")

for ax in axes:
    ax.set_xlabel("predicted"); ax.set_ylabel("true")

plt.tight_layout()
plt.savefig(OUT / "figures" / "15_test_confusion.png", dpi=150)
plt.show()

## 6. 오분류 쌍 순위

In [ ]:
pairs = []
for i in range(N_CLASSES):
    for j in range(N_CLASSES):
        if i != j and cm[i, j] > 0:
            pairs.append({"true": classes[i], "predicted": classes[j],
                          "count": int(cm[i, j]),
                          "pct_of_true": round(100 * cm[i, j] / cm[i].sum(), 1)})

pairs = pd.DataFrame(pairs).sort_values("count", ascending=False)
pairs.to_csv(OUT / "metrics" / "test_confusion_pairs.csv",
             index=False, encoding="utf-8")

print(f"전체 오분류 {int(cm.sum() - np.trace(cm))}건 / {len(y_true)}건 "
      f"({100 * (1 - np.trace(cm) / cm.sum()):.1f}%)")
print("\n상위 12개 혼동 쌍")
print(pairs.head(12).to_string(index=False))

## 7. EDA 가설 검증 — 유리 3종

EDA에서 계산한 Hue 히스토그램 겹침 계수는 이랬습니다.

| 쌍 | 겹침 | 예측 |
|---|---|---|
| brown ↔ green | 0.146 | 잘 구분됨 |
| green ↔ white | 0.202 | 잘 구분됨 |
| brown ↔ white | 0.422 | **가장 헷갈릴 것** |

색으로 구분 가능하다는 가설이 맞았다면 유리 3×3 블록의 대각선이 진해야 하고,
오분류가 있다면 brown↔white에 몰려야 합니다.

In [ ]:
glass_idx = [i for i, c in enumerate(classes) if "glass" in c]
glass_cls = [classes[i] for i in glass_idx]

sub  = cm[np.ix_(glass_idx, glass_idx)]
subn = sub / cm[glass_idx].sum(axis=1, keepdims=True)   # 분모는 전체 예측 기준

fig, ax = plt.subplots(figsize=(5.5, 4.5))
sns.heatmap(subn, annot=True, fmt=".3f", cmap="Blues", vmin=0, vmax=1,
            square=True, xticklabels=glass_cls, yticklabels=glass_cls, ax=ax)
ax.set_title("Glass 3x3 block (row-normalized)")
ax.set_xlabel("predicted"); ax.set_ylabel("true")
plt.tight_layout()
plt.savefig(OUT / "figures" / "16_glass_block.png", dpi=150)
plt.show()

overlap_eda = {("brown-glass", "green-glass"): 0.146,
               ("green-glass", "white-glass"): 0.202,
               ("brown-glass", "white-glass"): 0.422}

print("EDA 겹침 계수 vs 실제 상호 오분류")
print("-" * 56)
for (a, b), ov in sorted(overlap_eda.items(), key=lambda kv: kv[1]):
    ia, ib = classes.index(a), classes.index(b)
    both = int(cm[ia, ib] + cm[ib, ia])
    denom = int(cm[ia].sum() + cm[ib].sum())
    print(f"{a:12s} <-> {b:12s}  겹침 {ov:.3f}  |  "
          f"상호 오분류 {both:3d}건 ({100 * both / denom:.1f}%)")

**해석**

겹침 계수 순서와 상호 오분류 순서가 일치하면, EDA 단계의 색상 분석만으로
어떤 클래스가 헷갈릴지 예측할 수 있었다는 뜻입니다.
**데이터를 먼저 들여다본 것이 실제로 결과를 예측했다**는 게 보고서에서 가장 좋은 구조입니다.

일치하지 않는다면 그것도 유효한 발견입니다. 색 이외의 요인(형태·질감·배경)이
더 크게 작용했다는 뜻이고, CNN이 색만 보는 게 아니라는 증거가 됩니다.

## 8. 최악 클래스 심층 분석

In [ ]:
worst = per_class.iloc[0]["class"]
wi = classes.index(worst)

print(f"최저 F1 클래스: {worst} (F1 {per_class.iloc[0]['f1']:.3f}, "
      f"n={int(per_class.iloc[0]['n'])})")
print()
print(f"[{worst}]를 무엇으로 잘못 예측했나 (recall 손실)")
row = pd.DataFrame({"predicted": classes, "count": cm[wi]})
row = row[(row["count"] > 0) & (row["predicted"] != worst)].sort_values(
    "count", ascending=False)
row["pct"] = (100 * row["count"] / cm[wi].sum()).round(1)
print(row.to_string(index=False))

print()
print(f"무엇을 [{worst}]로 잘못 예측했나 (precision 손실)")
col = pd.DataFrame({"true": classes, "count": cm[:, wi]})
col = col[(col["count"] > 0) & (col["true"] != worst)].sort_values(
    "count", ascending=False)
col["pct"] = (100 * col["count"] / cm[:, wi].sum()).round(1)
print(col.to_string(index=False))

## 9. 오분류 이미지 육안 확인

In [ ]:
wrong = np.where(y_pred != y_true)[0]
order = wrong[np.argsort(-conf[wrong])]        # 확신하며 틀린 것부터

n_show = min(12, len(order))
fig, axes = plt.subplots(3, 4, figsize=(13, 10))

for ax, k in zip(axes.ravel(), order[:n_show]):
    ax.imshow(Image.open(test_df.loc[k, "path"]).convert("RGB"))
    ax.axis("off")
    ax.set_title(f"true: {classes[y_true[k]]}\npred: {classes[y_pred[k]]} "
                 f"({conf[k]:.2f})", fontsize=9, color="#c0392b")

for ax in axes.ravel()[n_show:]:
    ax.axis("off")

plt.suptitle("가장 확신하며 틀린 12장", fontsize=13)
plt.tight_layout()
plt.savefig(OUT / "figures" / "17_worst_errors.png", dpi=130)
plt.show()

**이 그림을 반드시 눈으로 보세요.**

발표에서 가장 반응이 좋은 슬라이드이면서, 실제로 가장 많은 정보를 줍니다.
확인할 것:

- **라벨 자체가 애매한가** — 유리병 사진인데 `metal` 뚜껑이 크게 찍혔다든지.
  이건 모델 잘못이 아니라 데이터셋의 한계입니다.
- **배경이 지배적인가** — 물체보다 배경이 넓게 찍혀 배경으로 판단했을 수 있습니다.
- **사람도 헷갈리는가** — 사람이 봐도 모르겠다면 그 오류는 개선 여지가 없습니다.

"모델이 왜 틀렸는지 설명할 수 있다"가 정확도 숫자보다 점수가 높습니다.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.hist(conf[y_pred == y_true], bins=30, alpha=0.65, label="correct",
        color="#4a7ba7", density=True)
ax.hist(conf[y_pred != y_true], bins=30, alpha=0.65, label="incorrect",
        color="#c0392b", density=True)
ax.set_xlabel("prediction confidence (max softmax)")
ax.set_ylabel("density")
ax.set_title("Confidence distribution")
ax.legend()
plt.tight_layout()
plt.savefig(OUT / "figures" / "18_confidence.png", dpi=150)
plt.show()

print(f"정답 평균 확신도 : {conf[y_pred == y_true].mean():.3f}")
print(f"오답 평균 확신도 : {conf[y_pred != y_true].mean():.3f}")

for t in [0.9, 0.95, 0.99]:
    keep = conf >= t
    if keep.sum():
        print(f"확신도 {t:.2f} 이상만 채택 시 — 커버리지 "
              f"{100 * keep.mean():5.1f}%, 정확도 "
              f"{100 * (y_pred[keep] == y_true[keep]).mean():.2f}%")

**실무적 의미**

오답의 확신도가 정답보다 뚜렷하게 낮다면, **확신도 임계값으로 걸러내는 운용**이 가능합니다.
"확신도 0.95 미만은 사람이 검수" 같은 방식이죠. 자동 분류 시스템에서 실제로 쓰는 방법이고,
보고서의 "활용 방안" 항목에 넣을 수 있는 구체적인 제안입니다.

## 10. 전체 실험 요약

In [ ]:
final = pd.read_csv(OUT / "metrics" / "transfer_comparison.csv")
final = final.rename(columns={"val_acc": "val_acc(%)"})

test_row = pd.DataFrame([{
    "model": "resnet18_finetune (TEST)",
    "val_acc(%)": round(test_acc, 2),
    "macro_f1": round(test_macro_f1, 4),
    "min_recall": round(float(rec.min()), 3),
    "worst_class": classes[int(rec.argmin())],
    "recall_std": round(float(rec.std()), 3),
    "minutes": np.nan,
}])

report = pd.concat([final, test_row], ignore_index=True)
report.to_csv(OUT / "metrics" / "final_report.csv", index=False, encoding="utf-8")
print(report.to_string(index=False))

summary = {
    "test_accuracy":    round(float(test_acc), 2),
    "test_macro_f1":    round(float(test_macro_f1), 4),
    "test_weighted_f1": round(float(test_w_f1), 4),
    "val_macro_f1":     round(float(ck["val_macro_f1"]), 4),
    "n_test":           int(len(y_true)),
    "n_errors":         int((y_pred != y_true).sum()),
    "best_class":       per_class.iloc[-1]["class"],
    "worst_class":      per_class.iloc[0]["class"],
    "top_confusion":    f"{pairs.iloc[0]['true']} -> {pairs.iloc[0]['predicted']} "
                        f"({int(pairs.iloc[0]['count'])}건)",
}
with open(OUT / "metrics" / "final_summary.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

print()
for k, v in summary.items():
    print(f"  {k:18s} {v}")

In [ ]:
print(classification_report(y_true, y_pred, target_names=classes,
                            digits=3, zero_division=0))

In [ ]:
figs = sorted((OUT / "figures").glob("*.png"))
print(f"발표자료용 그림 {len(figs)}장 — {(OUT / 'figures').resolve()}")
for p in figs:
    print("  ", p.name)

---

## 발표 슬라이드 매핑

| 슬라이드 | 그림 | 말할 것 |
|---|---|---|
| 데이터 | `01_class_dist` | 15,515장 12클래스, 불균형 8.77배 |
| 데이터 품질 | `02_resolution`, `03_samples` | 손상 0, 팔레트 모드 34장, 중복 19장 제거 |
| **가설 수립** | `04_glass_hue`, `05_glass_sat_val` | 유리 3종은 색으로 갈린다 — 겹침 계수로 정량화 |
| 분할 | `06_split` | stratified 70/15/15, leakage 0 |
| 불균형 실험 | `07_recall_by_strategy` | 정확도 -3.2p, 최악 recall +68% |
| 전이학습 | `11_transfer_recall`, `12_transfer_curves` | scratch 61% → frozen 92% (backbone 미학습) |
| **최종 결과** | `15_test_confusion` | test 성능, 혼동 구조 |
| **가설 검증** | `16_glass_block` | EDA 예측이 맞았는가 |
| 오류 분석 | `17_worst_errors` | 왜 틀렸는지 설명 |
| 활용 방안 | `18_confidence` | 확신도 임계값 운용 |

**이야기의 뼈대**: 데이터를 들여다봤고(EDA) → 가설을 세웠고(색상 분석) →
설계에 반영했고(증강·분할·가중치) → 검증했다(혼동행렬). 정확도 숫자 하나보다
이 흐름이 훨씬 강합니다.